In [1]:
import os
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

# 1. Chargement des variables d'environnement (ton fichier .env)
load_dotenv()

# Configuration Base de données (RDS) - On garde tes noms de variables
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME")

# 2. Initialisation du moteur SQLAlchemy pour PostgreSQL
connection_url = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_url)

def refresh_views(dossier_sql="scripts_sql"):
    """
    Lit et exécute tous les scripts SQL du dossier pour mettre à jour les vues.
    """
    # Chemin absolu pour éviter les erreurs sous Windows
    script_dir = os.getcwd()
    chemin_complet = os.path.abspath(os.path.join(script_dir, dossier_sql))
    
    if not os.path.exists(chemin_complet):
        print(f"❌ [ERREUR] Le dossier '{dossier_sql}' est introuvable ici : {chemin_complet}")
        return

    # On récupère les fichiers .sql
    fichiers_sql = [f for f in os.listdir(chemin_complet) if f.endswith('.sql')]
    
    # --- IMPORTANT : L'ordre d'exécution ---
    # On s'assure de créer view_caract en premier car les autres en dépendent souvent
    ordre_prioritaire = ['view_caract.sql', 'view_lieux.sql', 'view_vehicules.sql', 'view_usager.sql']
    
    # On trie la liste pour suivre cet ordre, puis on ajoute les autres fichiers s'il y en a
    fichiers_a_exécuter = [f for f in ordre_prioritaire if f in fichiers_sql]
    fichiers_restants = [f for f in fichiers_sql if f not in ordre_prioritaire]
    fichiers_finaux = fichiers_a_exécuter + fichiers_restants

    print(f"🚀 Connexion à la base RDS : {DB_HOST}")
    print(f"📄 Scripts détectés : {fichiers_finaux}\n")

    with engine.connect() as conn:
        for nom_fichier in fichiers_finaux:
            try:
                chemin_fichier = os.path.join(chemin_complet, nom_fichier)
                with open(chemin_fichier, 'r', encoding='utf-8') as f:
                    # On nettoie le SQL pour éviter les caractères parasites
                    sql_content = f.read().strip()
                
                if sql_content:
                    print(f"⏳ Exécution de : {nom_fichier}...")
                    # Utilisation de conn.begin() pour garantir que le script est validé (COMMIT)
                    with conn.begin():
                        conn.execute(text(sql_content))
                    print(f"✅ Vue mise à jour : {nom_fichier}")
                else:
                    print(f"⚠️ {nom_fichier} est vide, ignoré.")
                    
            except Exception as e:
                print(f"❌ Erreur sur {nom_fichier} : {str(e)[:200]}")

if __name__ == "__main__":
    refresh_views()

🚀 Connexion à la base RDS : db-securite-routiere.cz02sgc6ajv4.eu-north-1.rds.amazonaws.com
📄 Scripts détectés : ['view_caract.sql', 'view_lieux.sql', 'view_vehicules.sql', 'view_usager.sql']

⏳ Exécution de : view_caract.sql...
✅ Vue mise à jour : view_caract.sql
⏳ Exécution de : view_lieux.sql...
✅ Vue mise à jour : view_lieux.sql
⏳ Exécution de : view_vehicules.sql...
✅ Vue mise à jour : view_vehicules.sql
⏳ Exécution de : view_usager.sql...
✅ Vue mise à jour : view_usager.sql
